# Optuna Results Viewer
PyCharmのこのセルを実行すると、Optunaのデータベースから結果を取得し、インタラクティブなテーブルとして表示します。
`target_study_id` または `target_study_name` を変更することで、動的に読み込むStudyを切り替えられます。

In [1]:
import optuna
import pandas as pd
import sqlite3

# ==========================================
# 設定: ここを書き換えて動的にStudyを指定します
# ==========================================
db_path = "optuna_qrc_nonetype.db"

# Studyの指定方法（どちらかを使用。両方Noneなら一番最新を取得します）
target_study_id = 53    # 例: 1 (データベース内のIDで指定する場合)
target_study_name = None  # 例: "qrc_lorenz_vpt..." (名前で指定する場合)

# フィルタリング設定: value（VPTなど）がこの値以上の行のみ表示します
min_value_threshold = 8.0 # 例: 5.0
# ==========================================

storage_url = f"sqlite:///{db_path}"

# 利用可能なStudy一覧を取得するヘルパー関数
def get_studies(db_file):
    try:
        with sqlite3.connect(db_file) as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT study_id, study_name FROM studies ORDER BY study_id ASC")
            return cursor.fetchall()
    except Exception as e:
        return []

studies = get_studies(db_path)

if not studies:
    print(f"データベース '{db_path}' にStudyが見つからない、またはファイルが存在しません。")
    df_view = pd.DataFrame({"Error": ["Studyが見つかりません"]})
else:
    print("【利用可能なStudy一覧】")
    for s_id, s_name in studies:
        print(f"  ID: {s_id} | Name: {s_name}")
        
    # どのStudyを読み込むか決定
    selected_name = None
    if target_study_id is not None:
        for s_id, s_name in studies:
            if s_id == target_study_id:
                selected_name = s_name
                break
        if selected_name is None:
            print(f"\n⚠️ 警告: 指定された ID '{target_study_id}' は見つかりませんでした。")
            
    elif target_study_name is not None:
        for s_id, s_name in studies:
            if s_name == target_study_name:
                selected_name = s_name
                break
        if selected_name is None:
            print(f"\n⚠️ 警告: 指定された Name '{target_study_name}' は見つかりませんでした。")

    # 指定がない、または見つからなかった場合は最新（最後）のものを選択
    if selected_name is None:
        selected_name = studies[-1][1]
        print(f"\n-> 🔄 最新のStudyを読み込みます: {selected_name}")
    else:
        print(f"\n-> ✅ 指定されたStudyを読み込みます: {selected_name}")

    # Optunaからデータを取得
    study = optuna.load_study(study_name=selected_name, storage=storage_url)
    df = study.trials_dataframe()
    
    # 見やすいようにフォーマット（COMPLETEのみ抽出し、VPTで降順ソート）
    if 'state' in df.columns:
        df = df[df['state'] == 'COMPLETE']
    if 'value' in df.columns:
        # 指定された閾値以上のものだけをフィルタリング
        if min_value_threshold is not None:
            df = df[df['value'] >= min_value_threshold]
        df = df.sort_values(by='value', ascending=False)
        
    # 不要な列を隠して見やすくする（datetime等）
    drop_cols = [c for c in df.columns if c.startswith('datetime_') or c.startswith('duration')]
    df_view = df.drop(columns=drop_cols)
    
    # 必要な列を抽出してCSVに保存
    import os
    export_cols = ['value', 'params_feature_max', 'params_feedback_scale', 'params_leak_rate']
    valid_export_cols = [c for c in export_cols if c in df_view.columns]
    if valid_export_cols:
        out_path = os.path.join("/home/yoshi/PycharmProjects/Reservoir/benchmarks", "filtered_optuna_results.csv")
        df_view[valid_export_cols].to_csv(out_path, index=False)
        print(f"\n✅ 抽出したデータを '{out_path}' に保存しました。")

# DataFrameを表示（PyCharmが綺麗なテーブルでレンダリングします）
df_view.head(50)

/home/yoshi/PycharmProjects/Reservoir/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


【利用可能なStudy一覧】
  ID: 8 | Name: qrc_vpt_minmax_nonetype_q10_Z+ZZ_poly_square
  ID: 9 | Name: qrc_vpt_minmax_nonetype_q16_Z+ZZ_poly_square
  ID: 10 | Name: qrc_vpt_minmax_nonetype_q8_Z+ZZ_poly_square
  ID: 11 | Name: qrc_vpt_minmax_nonetype_q6_Z+ZZ_poly_square
  ID: 15 | Name: qrc_vpt_crs_centered_nonetype_q10_Z+ZZ_poly_square
  ID: 16 | Name: qrc_vpt_crs_nonetype_q10_Z+ZZ_poly_square
  ID: 17 | Name: qrc_vpt_crs_centered_nonetype_q6_Z+ZZ_poly_square
  ID: 19 | Name: qrc_vpt_affine_nonetype_q6_Z+ZZ_poly_square
  ID: 20 | Name: qrc_vpt_affine_nonetype_q10_Z+ZZ_poly_square
  ID: 21 | Name: qrc_vpt_affine_nonetype_q16_Z+ZZ_poly_square
  ID: 22 | Name: qrc_vpt_minmax_nonetype_q6_Z+ZZ_poly_square_kai
  ID: 23 | Name: qrc_vpt_minmax_nonetype_q16_Z+ZZ_poly_square_kai
  ID: 24 | Name: qrc_vpt_minmax0_nonetype_q16_Z+ZZ_poly_square_kai
  ID: 25 | Name: qrc_vpt_minmax0_nonetype_q6_Z+ZZ_poly_square_kai2
  ID: 28 | Name: qrc_vpt_minmax0_nonetype_q16_Z+ZZ_poly_square_kai4
  ID: 32 | Name: qrc_vpt_minm

,number,value,params_feature_max,params_feature_min,params_feedback_scale,params_leak_rate,params_n_layers,user_attrs_best_lambda,user_attrs_correlation,user_attrs_error,...,user_attrs_status,user_attrs_truth_max,user_attrs_truth_mean,user_attrs_truth_min,user_attrs_truth_std,user_attrs_var_ratio,user_attrs_vpt_lt,user_attrs_vpt_steps,user_attrs_vpt_threshold,state
193,193,8.854545,0.051530,-0.000097,0.325031,0.252866,1,1.000000e-12,0.782014,NaN,...,success,0.053765,0.025293,-0.002092,0.010063,0.920752,8.854545,974.0,0.4,COMPLETE
332,332,8.845455,0.075991,-0.000097,0.120664,0.164332,1,1.000000e-12,0.839593,NaN,...,success,0.079285,0.037323,-0.003036,0.014831,0.989046,8.845455,973.0,0.4,COMPLETE
246,246,8.827273,0.056875,-0.000097,0.222336,0.167102,1,1.000000e-12,0.807229,NaN,...,success,0.059341,0.027921,-0.002298,0.011105,0.941054,8.827273,971.0,0.4,COMPLETE
345,345,8.827273,0.090816,-0.000097,0.000015,0.537984,1,1.000000e-12,0.782499,NaN,...,success,0.094751,0.044613,-0.003609,0.017720,0.887579,8.827273,971.0,0.4,COMPLETE
212,212,8.800000,0.065850,-0.000097,0.186972,0.220787,1,1.000000e-12,0.839333,NaN,...,success,0.068705,0.032335,-0.002645,0.012854,0.950288,8.800000,968.0,0.4,COMPLETE
275,275,8.718182,0.059555,-0.000097,0.010643,0.532282,1,1.000000e-12,0.773570,NaN,...,success,0.062138,0.029240,-0.002402,0.011627,0.884290,8.718182,959.0,0.4,COMPLETE
113,113,8.700000,0.059647,-0.000097,0.000944,0.535538,1,1.000000e-12,0.768211,NaN,...,success,0.062233,0.029284,-0.002405,0.011645,0.863634,8.700000,957.0,0.4,COMPLETE
217,217,8.700000,0.051285,-0.000097,0.369830,0.199374,1,1.000000e-12,0.762570,NaN,...,success,0.053510,0.025172,-0.002082,0.010015,0.890810,8.700000,957.0,0.4,COMPLETE
346,346,8.690909,0.056691,-0.000097,0.017914,0.524296,1,1.000000e-12,0.833258,NaN,...,success,0.059150,0.027831,-0.002291,0.011069,0.978130,8.690909,956.0,0.4,COMPLETE
117,117,8.690909,0.070033,-0.000097,0.110508,0.587595,1,1.000000e-12,0.766104,NaN,...,success,0.073068,0.034392,-0.002806,0.013669,0.966415,8.690909,956.0,0.4,COMPLETE
